In [ ]:
%matplotlib inline

# Session 1: Differentiable Programming with HIPS/Autograd

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Set up codespace now. (It will take a while!)</b>

We will show you where to find this notebook in the repo while you wait.

</div>

## Motivation

Differentiable programming is an enabling technology. Given scientific code for computing some quantity, it allows us to generate derivatives of quantities involved in the computation without any need for deriving or hand-coding the derivative expressions involved.

Some motivating examples:
* Computing the **Jacobian** for a nonlinear system.
* Computing the **gradient** required for an ODE- or PDE-constrained optimisation method.
* The **backpropagation** operation used for training machine learning models.
* Computing **Hessians** for uncertainty quantification methods.
* Solving the **adjoint** problems involved in data assimilation methods commonly used for weather forecasting.

## Learning objectives

In today's session we will:

* Get a brief history of automatic differentiation.
* Learn about *forward mode* and *reverse mode*.
* Learn about the *operator overloading* approach.
* Try out the *Autograd* AD tool applied to some test problems.
* Verify the code generated by Autograd both manually and using the *Taylor test*.

## Preparations

#### Terminology

This course introduces the concept of *differentiable programming*, a.k.a. *automatic differentiation (AD)*, or *algorithmic differentiation*. We will use the acronym AD henceforth.

#### Notation

For a differentiable *mathematical* function $f:A\rightarrow\mathbb{R}$ with scalar input (i.e., a single value) from $A\subseteq\mathbb{R}$, we make use of both the Lagrange notation $f'(x)$ and Leibniz notation $\frac{\mathrm{d}f}{\mathrm{d}x}$ for its derivative.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Caution</b> with the physics notation for derivatives $\dot{x}$. It won't always mean what you expect! (See later.)
</div>

Similarly, for $m\in\mathbb{N}$ dimensional, differentiable, vector-valued function $\mathbf{f}:A\rightarrow\mathbb{R}^m$ with scalar input, we have derivative notations $\mathbf{f}'(x)$ and $\frac{\mathrm{d}\mathbf{f}}{\mathrm{d}x}$.

For a differentiable function with vector input (i.e., multiple inputs), we use partial derivative notation. For example, if $f:\mathbb{R}^2\rightarrow\mathbb{R}$ is written as $f=f(x,y)$ then we have the partial derivatives $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$ with respect to first and second components, respectively. We use
$$\nabla f=\left(\frac{\partial f}{\partial x_1},\dots,\frac{\partial f}{\partial x_m}\right)$$
to denote the vector of all such partial derivatives. Similarly for vector-valued functions with multiple inputs.

When it comes to derivatives in code, we use the `_d` notation (for "derivative" or "dot"), which is standard in the AD literature. Its meaning will be described in due course.

## History

* Origins of AD in 1950s.
* However, it found a wider audience in the 1980s, when it became more relevant thanks to advances in both computer power and modern programming languages.
* Forward mode (the subject of this session) was discovered by Wengert in 1964.
* Further developed by Griewank in the late 1980s.

<div style="text-align: center;">
  <img src="images/Wengert.png" width="600" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 1:</strong> Header of <a href="https://doi.org/10.1145/355586.364791">(R. E. Wengert, 1964)</a>.
</div>

## Idea

The idea of AD is to **treat a model as a sequence of elementary instructions** (e.g., addition, multiplication, exponentiation). Here a *model* could be a function or subroutine, code block, or a whole program. Elementary operations are well-understood and their derivatives are known. As such, the derivative of the whole model may be computed by composing the derivatives of each operation using the *chain rule*.

#### Recap on A-level maths: the Chain Rule

Consider two composable, differentiable (mathematical) functions, $f$ and $g$, with composition $h=f\circ g$. By definition, this means
$$h(x)=(f\circ g)(x)=g(f(x)).$$

Then the *chain rule* states that the derivative of $h$ may be computed in terms of the derivatives of $f$ and $g$ using the formula
$$h'(x)=(f\circ g)'(x)=(f\circ g')(x)\,f'(x)=g'(f(x))\,f'(x).$$

Equivalently, in Leibniz notation:
$$\frac{\mathrm{d}h}{\mathrm{d}x}=\frac{\mathrm{d}g}{\mathrm{d}f}\frac{\mathrm{d}f}{\mathrm{d}x}.$$

For variables with multiple arguments, the result is equivalent for each partial derivative, e.g.,
$$\frac{\partial h}{\partial x}=\frac{\partial g}{\partial f}\frac{\partial f}{\partial x}.$$

## $f$ \& $g$ example: Directed Acyclic Graph

We can visualise the functions in terms of DAGs.

Recalling that
$$f(x_1,x_2)=x_1x_2$$
and
$$g(y)=(\sin(y),\cos(y))\, ,$$
we have

<div style="text-align: center;">
  <img src="images/f_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 2:</strong> Directed Acyclic Graph (DAG) for the $f$ function in the $f$ & $g$ example. Generated using tikZ and $\LaTeX$.
</div>

<div style="text-align: center;">
  <img src="images/g_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 3:</strong> Directed Acyclic Graph (DAG) for the $g$ function in the $f$ & $g$ example. Generated using tikZ and $\LaTeX$.
</div>

## $f$ \& $g$ example: composition

Consider the composition $h=f\circ g:\mathbb{R}^2\rightarrow\mathbb{R}^2$, which is given by
$$h(x_1,x_2)=(f\circ g)(x_1,x_2)=g(f(x_1,x_2))=g(x_1x_2)=(\sin(x_1x_2),\cos(x_1x_2)).$$

For the derivative of each component,
$$
\frac{\partial f}{\partial x_1}=\frac{\partial}{\partial x_1}x_1x_2=x_2,
\quad\frac{\partial f}{\partial x_2}=\frac{\partial}{\partial x_2}x_1x_2=x_1,
\quad\frac{\partial g}{\partial y}=\frac{\partial}{\partial y}(\sin(y),\cos(y))=(\cos(y),-\sin(y)).
$$

Introduce the notation $g(y)=(g_1(y),g_2(y))$ so that
$$
\frac{\partial g_1}{\partial y}=\cos(y),
\quad\frac{\partial g_2}{\partial y}=-\sin(y).
$$
Similarly $h(x_1,x_2)=(h_1(x_1,x_2),h_2(x_1,x_2))=(g_1(f(x_1,x_2)),g_2(f(x_1,x_2)))$.

Let's use the chain rule to work out the derivatives of each of the outputs with respect to each of the inputs.
$$
\frac{\partial h_1}{\partial x_1}=\frac{\partial g_1}{\partial f}\frac{\partial f}{\partial x_1}=\cos(y)x_2=x_2\cos(x_1x_2),
\quad\frac{\partial h_1}{\partial x_2}=\frac{\partial g_1}{\partial f}\frac{\partial f}{\partial x_2}=\cos(y)x_1=x_1\cos(x_1x_2),
$$
where $y=f(x_1,x_2)$ and
$$
\quad\frac{\partial h_2}{\partial x_1}=\frac{\partial g_2}{\partial f}\frac{\partial f}{\partial x_1}=-\sin(y)x_2=-x_2\sin(x_1x_2),
\quad\frac{\partial h_2}{\partial x_2}=\frac{\partial g_2}{\partial f}\frac{\partial f}{\partial x_2}=-\sin(y)x_1=-x_1\sin(x_1x_2).
$$

We will come back to these formulae to verify the correctness of our AD computations.

## $f$ \& $g$ example: Seed vectors

Let's revisit the DAG interpretation and consider how the derivatives work.

<div style="text-align: center;">
  <img src="images/forward_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 4:</strong> Directed Acyclic Graph (DAG) for the composition of the functions in the $f$ & $g$ example.
</div>